# Analysis: Stability and Latent Space

This notebook consolidates the stability experiments and latent space visualizations.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.models import DeepONet
from src.data_loader import MarketDataLoader

## Stability Analysis (Lipschitz Continuity)
We test if small perturbations in the input history result in bounded changes in the latent variables.

In [ ]:
def stability_test():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DeepONet(input_channels=6, latent_dim=16).to(device)
    
    # Load weights if available
    model_path = '../models/deeponet.pth'
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print("Loaded model.")
    else:
        print("Using random weights.")
    model.eval()
    
    # Generate sample input (Batch, Seq, Chan)
    x_true = torch.randn(1, 30, 6).to(device)
    
    # Forward pass to get latent (B is latent coefficients)
    # DeepONet returns (prices, b)
    
    # Note: DeepONet forward expects (x, grid); this analysis only needs the branch output.
    # Call model.branch_cnn directly instead of running the full forward pass.
    
    # Only accessing branch:
    x_perm = x_true.permute(0, 2, 1) # (1, 6, 30)
    features = model.branch_cnn(x_perm)
    b_true = model.branch_mlp(features)
    
    noise_levels = np.linspace(0, 0.5, 20)
    deviations = []
    
    for sigma in noise_levels:
        noise = torch.randn_like(x_true) * sigma
        x_noisy = x_true + noise
        
        x_noisy_perm = x_noisy.permute(0, 2, 1)
        feat_noisy = model.branch_cnn(x_noisy_perm)
        b_noisy = model.branch_mlp(feat_noisy)
        
        dist = torch.norm(b_noisy - b_true).item()
        deviations.append(dist)
        
    plt.figure(figsize=(8, 5))
    plt.plot(noise_levels, deviations, marker='o')
    plt.xlabel("Input Noise $\\sigma$")
    plt.ylabel("Latent Deviation $||z - z_{true}||$")
    plt.title("Stability of Neural Operator Encoder")
    plt.grid(True)
    plt.show()

stability_test()